Machine Learning for Scientists: Topic 3 - Classification with k-NN and Naïve Bayes. This notebook explores two distinct foundational paradigms for predicting categorical class labels:   k-Nearest Neighbors (k-NN): A instance-based, non-parametric method that classifies samples based on proximity in feature space.   Naïve Bayes: A generative probabilistic model that computes $P(Y \mid X)$ by making the simplifying assumption that features are conditionally independent given the class label.   

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
from matplotlib.colors import ListedColormap

# Set random seed
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries imported successfully.")

Step 1: Synthesize Binary Classification Data. To evaluate both classifiers, we synthesize a 2D dataset representing two distinct patient or cellular classes. Two continuous features represent biological measurements.   Non-linear Boundaries: k-NN excels at capturing local, non-linear cluster shapes without assuming a specific data distribution.   Feature Distributions: Gaussian Naïve Bayes fits continuous data by assuming features within each class follow independent Gaussian distributions.  

In [ ]:
# Generate synthetic 2D binary classification dataset
X, y = make_classification(
    n_samples=200,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=42
)

# Train/Test Split (80/20)
split = 160
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Plot Dataset
plt.figure(figsize=(7, 5))
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], color='red', label='Class 0 (Control)', alpha=0.8)
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], color='blue', label='Class 1 (Disease)', alpha=0.8)
plt.title("Synthetic Scientific Classification Dataset", fontsize=13)
plt.xlabel("Feature 1 (e.g., Biomarker A)", fontsize=11)
plt.ylabel("Feature 2 (e.g., Biomarker B)", fontsize=11)
plt.legend()
plt.show()

Step 2: k-Nearest Neighbors (k-NN) & Impact of Hyperparameter $k$-NN assigns an unseen observation $x$ to the majority class among its $k$ closest neighbors in Euclidean space:   Small $k$ (e.g., $k=1$): Creates complex, highly flexible boundaries that can overfit to local measurement noise.   Large $k$ (e.g., $k=30$): Produces smooth decision boundaries that reduce variance, but can over-smooth and miss fine structural details.   Below, we observe how varying $k$ shifts the model from high variance (overfitting) to high bias (underfitting).

In [ ]:
def plot_decision_boundary(clf, X, y, title, ax):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                         np.arange(y_min, y_max, 0.05))

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
    cmap_bold = ['red', 'blue']

    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)
    for c, color in zip([0, 1], cmap_bold):
        ax.scatter(X[y == c, 0], X[y == c, 1], c=color, label=f'Class {c}', edgecolors='k', alpha=0.7)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

k_values = [1, 7, 35]
for idx, k in enumerate(k_values):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    plot_decision_boundary(knn, X_train, y_train, f"k-NN (k={k}) - Train Acc: {knn.score(X_train, y_train):.2f}", axes[idx])

plt.tight_layout()
plt.show()

Step 3: Naïve Bayes & Conditional Independence Naïve Bayes computes class probabilities using Bayes' rule:   $$P(Y = c \mid X) \propto P(Y = c) \prod_{j=1}^p P(X_j \mid Y = c) \quad \text{[cite: 25]}$$Core Assumption: Features $X_1, X_2, \dots, X_p$ are conditionally independent given the class label $Y$.   Parameter Reduction: Instead of estimating a full $p$-dimensional joint density, it estimates $p$ simple 1D distributions per class, preventing parameter explosion.   Gaussian Naïve Bayes: Fits a 1D normal distribution $\mathcal{N}(\mu_{c,j}, \sigma_{c,j}^2)$ for each feature $j$ given class $c$.   

In [ ]:
# Fit Gaussian Naïve Bayes
gnb = GaussianNB()
gnb.fit(X_train, y_train)

# Compare Decision Boundaries between k-NN (k=7) and Naïve Bayes
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

knn_opt = KNeighborsClassifier(n_neighbors=7).fit(X_train, y_train)

plot_decision_boundary(knn_opt, X_train, y_train, f"k-NN (k=7)\nTest Acc: {accuracy_score(y_test, knn_opt.predict(X_test)):.2f}", axes[0])
plot_decision_boundary(gnb, X_train, y_train, f"Gaussian Naïve Bayes\nTest Acc: {accuracy_score(y_test, gnb.predict(X_test)):.2f}", axes[1])

plt.tight_layout()
plt.show()

Step 4: Model Comparison & Evaluation Metrics. We evaluate performance on the held-out test set:   Classifier Accuracy: Measures overall prediction correctness.Robustness: Naïve Bayes provides fast, parametric probability estimates that scale well to high dimensions , whereas k-NN requires storing and searching all training data at test time.

In [ ]:
knn_preds = knn_opt.predict(X_test)
gnb_preds = gnb.predict(X_test)

print("=== k-Nearest Neighbors (k=7) Classification Report ===")
print(classification_report(y_test, knn_preds, target_names=['Control (0)', 'Disease (1)']))

print("=== Gaussian Naïve Bayes Classification Report ===")
print(classification_report(y_test, gnb_preds, target_names=['Control (0)', 'Disease (1)']))